**food calories**

In [ ]:
import pandas as pd

df1 = pd.read_csv("food_data-calories.csv")
df2 = pd.read_csv("large_food_dataset (3).csv")

print("df1:")
print(df1.head())

print("\ndf2:")
print(df2.head())

df1:
      Food Name Serving Calories         Category
0  Baked Potato   100 g  122 Cal  potato-products
1    Croquettes   100 g  180 Cal  potato-products
2   Curly Fries   100 g  176 Cal  potato-products
3  French Fries   100 g  312 Cal  potato-products
4       Gnocchi   100 g  163 Cal  potato-products

df2:
      name  calories  protein    fat   carbs
0  CHICKEN     107.0    21.40   1.79    0.00
1  CHICKEN     167.0     6.67   0.00   33.30
2  CHICKEN     188.0    24.70   9.41    1.18
3  CHICKEN     556.0     0.00   0.00  111.00
4  CHICKEN     219.0    15.60  15.60    9.38


In [ ]:
df1.rename(columns={"Food Name": "name"}, inplace=True)

print(df1.columns)

Index(['name', 'Serving', 'Calories', 'Category'], dtype='object')


In [ ]:
import re

def clean(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.strip()

df1["name"] = df1["name"].apply(clean)
df2["name"] = df2["name"].apply(clean)

In [ ]:
df1 = df1.dropna()
df2 = df2.dropna()

In [ ]:
df1 = df1.drop_duplicates()
df2 = df2.drop_duplicates()

In [ ]:
merged = pd.merge(df1, df2, on="name", how="inner")

print(merged.head())
print("Rows:", len(merged))

                    name Serving Calories         Category  calories  protein  \
0           potato flour   100 g  361 Cal  potato-products    1490.0     6.90   
1         potato pancake   100 g  268 Cal  potato-products     196.0     4.47   
2  potato salad with egg   100 g  250 Cal  potato-products     658.0     1.96   
3           sweet potato   100 g  116 Cal  potato-products     106.0     5.45   
4           sweet potato   100 g  116 Cal  potato-products     250.0     6.25   

     fat  carbs  
0   0.34  83.10  
1  10.80  20.64  
2   9.40  16.20  
3   1.28  16.40  
4  12.50  32.80  
Rows: 720


In [ ]:
!pip install rapidfuzz

In [ ]:
from rapidfuzz import process

choices = df2["name"].tolist()

def match_name(x):
    match = process.extractOne(x, choices)
    if match and match[1] > 80:
        return match[0]
    return None

df1["matched_name"] = df1["name"].apply(match_name)

merged = pd.merge(df1, df2, left_on="matched_name", right_on="name")

In [ ]:
print(merged.columns)

Index(['name_x', 'Serving', 'Calories', 'Category', 'matched_name', 'name_y',
       'calories', 'protein', 'fat', 'carbs'],
      dtype='object')


In [ ]:
merged.to_csv("merged_food_data.csv", index=False)

print("Saved ✅")

Saved ✅


In [ ]:
# لو لسه ما عملتيش clean
merged["Calories_clean"] = merged["Calories"].str.replace(" Cal", "").astype(float)

# نجيب أقل calories لكل food
idx = merged.groupby("name_x")["Calories_clean"].idxmin()

# نفلتر الداتا
merged_clean = merged.loc[idx]

# reset index
merged_clean = merged_clean.reset_index(drop=True)

In [ ]:
print(merged_clean["name_x"].duplicated().sum())

0


In [ ]:
merged_clean.to_csv("cleaned_final_data.csv", index=False)

In [ ]:
# 1) طلّعي الرقم من عمود Calories (زي "209 Cal" -> 209)
merged_clean["calories"] = (
    merged_clean["Calories"]
    .str.extract(r'(\d+)')        # يستخرج الرقم
    .astype(float)
)

# 2) احذفي الأعمدة اللي مش عايزاها
merged_clean.drop(
    columns=["Calories", "Calories_clean"],
    inplace=True,
    errors="ignore"
)

# 3) rename الأعمدة المطلوبة
merged_clean.rename(columns={
    "name_x": "name",
    "Serving": "portion",
    "Category": "category",
    "carbs": "carb"
}, inplace=True)

# 4) ترتيب الأعمدة النهائي
final_df = merged_clean[[
    "name",
    "portion",
    "category",
    "matched_name",
    "calories",   # ده دلوقتي = 209 صح
    "protein",
    "fat",
    "carb"
]]

# 5) حفظ
final_df.to_csv("final_cleaned_data.csv", index=False)

In [ ]:
final_df = merged_clean.rename(columns={
    "name_x": "name",
    "Serving": "portion",
    "Category": "category"
})

In [ ]:
final_df = final_df[[
    "name",
    "portion",
    "category",
    "matched_name",
    "calories",
    "protein",
    "fat",
    "carbs"
]]

In [ ]:
final_df.to_csv("final_cleaned_data.csv", index=False)